# Juego del Nim

En esta práctica implementaremos el juego del Nim. El juego consiste en varias pilas de fichas. En cada turno, un jugador debe retirar fichas de una de las pilas. Pierde quien se queda sin fichas. En esta práctica implementaremos una versión simplificada con una sola pila.

La pila empieza con 20 fichas y, en cada turno, se pueden retirar 1, 2 o 3. Pierde quien se queda sin fichas. Por ejemplo, si hay dos jugadores, el primero retira 3 fichas y el segundo retira 2, el primero pierde porque en su siguiente turno se quedará sin fichas.

## Implementación del juego

### Representación del estado

Primero necesitaremos una forma de representar los distintos estados del juego. En cada estado definiremos:

* `jugador`: jugador al que le toca realizar el siguiente movimiento.
* `utilitat`: utilidad del estado para el jugador.
* `tauler`: estado del tablero. En este caso, basta con conocer cuántas fichas quedan en la pila.
* `moviments`: movimientos posibles desde el estado actual.
* `moviment`: movimiento realizado para llegar al estado actual.
* `estat_pare`: estado anterior al estado actual.

In [365]:
from dataclasses import dataclass


@dataclass(frozen=True)
class EstatJoc:
    torn: str
    utilitat: int
    tauler: int
    moviments: list
    moviment: int = None
    estat_pare: 'EstatJoc' = None

### Clase base `Joc`

Nuestros juegos heredarán de la clase `Joc`, que implementa los métodos básicos para programar juegos. Esta clase dispone de los siguientes métodos:

* `accions(self, estat)`: devuelve los movimientos posibles desde el estado recibido como parámetro.
* `resultat(self, estat, accio)`: devuelve el estado que resulta de aplicar la acción al estado actual.
* `es_terminal(self, estat)`: devuelve `True` si el estado es terminal.
* `utilitat(self, estat, jugador)`: devuelve la utilidad del estado para el jugador.

In [366]:
class Joc:
    """Un joc és similar a un problema, però té una utilitat per a cada estat i un test terminal en lloc d'un cost de ruta i un test de l'objectiu. Per crear un joc, subclasseu aquesta classe i implementeu accions, resultats, utilitat, i test_terminal. Podeu sobreescriure la visualització i els successors o bé podeu heretar els seus mètodes per defecte. També necessitareu establir l'atribut .inicial a l'estat inicial; això es pot fer en el constructor."""

    def accions(self, estat):
        """Retorna una llista dels moviments permesos en aquest punt."""
        raise NotImplementedError

    def resultat(self, estat, moviment):
        """Retorna l'estat que resulta de fer un moviment des d'un estat."""
        raise NotImplementedError

    def utilitat(self, estat, jugador):
        """Retorna el valor d'aquest estat final per al jugador."""
        raise NotImplementedError

    def test_terminal(self, estat):
        """Retorna True si aquest és un estat final per al joc."""
        return not self.accions(estat)

    def torn(self, estat):
        """Retorna el jugador al qual li toca moure en aquest estat."""
        return estat.torn

    def display(self, estat):
        """Imprimeix o mostra d'alguna manera l'estat."""
        print(estat)

    def __repr__(self):
        return '<{}>'.format(self.__class__.__name__)

    def juga_joc(self, *jugadors):
        """Juga un joc de n-persones, alternant torns."""
        estat = self.inicial
        self.display(estat)
        while True:
            for jugador in jugadors:
                moviment = jugador(self, estat)
                estat = self.resultat(estat, moviment)
                self.display(estat)
                if self.test_terminal(estat):
                    self.display(estat)
                    print()
                    return self.utilitat(estat, self.torn(self.inicial))

Crearemos una clase llamada `JocNim` que herede de `Joc`. Ten en cuenta estas condiciones:

* Los jugadores se turnan.
* El número inicial de fichas se define en el constructor de la clase; por defecto, será 20.
* Los movimientos posibles también se definen en el constructor; por defecto, se pueden retirar 1, 2 o 3 fichas.
* En cada turno se debe pedir al jugador cuántas fichas quiere retirar. Si el movimiento no es válido, hay que volver a pedir uno válido.
* La utilidad será 1 si gana `'MAX'` (es decir, si `'MIN'` fue el último en mover), -1 si gana `'MIN'` y 0 si la partida no ha terminado.
* El estado inicial tendrá como jugador a `'MAX'`; el tablero estará vacío y todas las fichas estarán disponibles.

In [367]:
class JocNim(Joc):
    def __init__(self, n_fitxes=20, moviments=[1, 2, 3]):
        self.inicial = EstatJoc(
            torn='MAX',
            utilitat=0,
            tauler=n_fitxes,
            moviments=moviments,
            moviment=None,
            estat_pare=None
        )

    def accions(self, estat):
        return [
            moviment for moviment in estat.moviments
            if moviment <= estat.tauler
        ]

    def resultat(self, estat, moviment):
        return EstatJoc(
            torn='MAX' if estat.torn == 'MIN' else 'MIN',
            utilitat=self.utilitat(estat, estat.torn),
            tauler=estat.tauler - moviment,
            moviments=estat.moviments,
            moviment=moviment,
            estat_pare=estat
        )

    def utilitat(self, estat, jugador):
        if estat.tauler == 0:
            return 1 if jugador != estat.torn else -1
        return 0

    def display(self, estat):
        print('Torn:', estat.torn)
        print('Tauler:', estat.tauler)
        print('Moviments:', self.accions(estat))
        print('Moviment:', estat.moviment)
        print('Utilitat:', estat.utilitat)
        print("")

### `JugadorAleatori`

Para jugar necesitaremos un jugador aleatorio. Este jugador elige al azar uno de los movimientos posibles.

In [368]:
import random


def jugador_aleatori(joc, estat):
    return random.choice(joc.accions(estat))

Comprobamos que el juego funciona con dos jugadores aleatorios.

In [369]:
joc = JocNim()
joc.juga_joc(jugador_aleatori, jugador_aleatori)

Torn: MAX
Tauler: 20
Moviments: [1, 2, 3]
Moviment: None
Utilitat: 0

Torn: MIN
Tauler: 19
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 18
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 17
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 15
Moviments: [1, 2, 3]
Moviment: 2
Utilitat: 0

Torn: MIN
Tauler: 12
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MAX
Tauler: 10
Moviments: [1, 2, 3]
Moviment: 2
Utilitat: 0

Torn: MIN
Tauler: 8
Moviments: [1, 2, 3]
Moviment: 2
Utilitat: 0

Torn: MAX
Tauler: 7
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 5
Moviments: [1, 2, 3]
Moviment: 2
Utilitat: 0

Torn: MAX
Tauler: 3
Moviments: [1, 2, 3]
Moviment: 2
Utilitat: 0

Torn: MIN
Tauler: 0
Moviments: []
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 0
Moviments: []
Moviment: 3
Utilitat: 0



1

## Búsqueda adversaria: minimax

Para que el juego resulte interesante, necesitamos un jugador inteligente. Implementaremos un jugador que utiliza el algoritmo minimax para decidir qué movimiento realizar.

Para implementar minimax necesitaremos una función de evaluación que mida la utilidad de un estado para un jugador. En este caso, la función de evaluación será la propia utilidad del estado. Devolverá un valor entre -1 y 1: 1 si el estado es ganador para `'MAX'`, -1 si es ganador para `'MIN'` y 0 si no es terminal.

También necesitaremos estas funciones auxiliares:

* `maximitza(joc, estat)`: devuelve la utilidad máxima que puede obtener `'MAX'` y el movimiento que la produce.
* `minimitza(joc, estat)`: devuelve la utilidad mínima para `'MAX'` y el movimiento que la produce.
* `minimax(joc, estat)`: devuelve el movimiento más conveniente para `'MAX'`, utilizando las funciones anteriores.

In [370]:
def minimax(joc, estat):
    def maximitza(joc, estat):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        v = -float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = minimitza(joc, joc.resultat(estat, accio))
            if v2 > v:
                v = v2
                moviment = accio
        return v, moviment

    def minimitza(joc, estat):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        v = float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = maximitza(joc, joc.resultat(estat, accio))
            if v2 < v:
                v = v2
                moviment = accio
        return v, moviment

    return maximitza(joc, estat)[1]

Comprobamos el funcionamiento de minimax con el juego del Nim.

In [371]:
joc = JocNim(n_fitxes=25)
joc.juga_joc(minimax, minimax)

Torn: MAX
Tauler: 25
Moviments: [1, 2, 3]
Moviment: None
Utilitat: 0
Torn: MIN
Tauler: 24
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0
Torn: MAX
Tauler: 21
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0
Torn: MIN
Tauler: 20
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0
Torn: MAX
Tauler: 17
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 16
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0
Torn: MAX
Tauler: 13
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 12
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 9
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 8
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 5
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 4
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 1
Moviments: [1]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 0
Moviments: []
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 0
Moviments: []
Moviment: 1
Utilitat: 0



1

## Búsqueda adversaria con poda alfa-beta

El algoritmo minimax es eficaz, pero resulta poco práctico cuando hay muchas posibilidades. Implementaremos la poda alfa-beta, que mejora minimax al reducir el número de nodos que hay que evaluar.

Para ello necesitaremos estas funciones:

* `maximitza(joc, estat, alfa, beta)`: devuelve el movimiento con mayor utilidad para `'MAX'`. `alfa` y `beta` son, respectivamente, los valores de la mejor opción encontrada hasta el momento para `'MAX'` y `'MIN'`.
* `minimitza(joc, estat, alfa, beta)`: devuelve el movimiento con menor utilidad para `'MAX'`; `alfa` y `beta` tienen el mismo significado.
* `minimax_ab(joc, estat)`: devuelve el movimiento con mayor utilidad para `'MAX'`.

In [372]:
def minimax_ab(joc, estat):
    def maximitza(joc, estat, alfa, beta):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        v = -float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = minimitza(joc, joc.resultat(estat, accio), alfa, beta)
            if v2 > v:
                v = v2
                moviment = accio
            if v >= beta:
                return v, moviment
            alfa = max(alfa, v)
        return v, moviment

    def minimitza(joc, estat, alfa, beta):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        v = float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = maximitza(joc, joc.resultat(estat, accio), alfa, beta)
            if v2 < v:
                v = v2
                moviment = accio
            if v <= alfa:
                return v, moviment
            beta = min(beta, v)
        return v, moviment

    return maximitza(joc, estat, -float('inf'), float('inf'))[1]

Para comprobar la poda alfa-beta, jugaremos una partida de 25 fichas entre un jugador aleatorio y otro que utiliza este algoritmo.

In [373]:
joc = JocNim(n_fitxes=25)
joc.juga_joc(minimax_ab, minimax_ab)

Torn: MAX
Tauler: 25
Moviments: [1, 2, 3]
Moviment: None
Utilitat: 0
Torn: MIN
Tauler: 24
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0
Torn: MAX
Tauler: 21
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0
Torn: MIN
Tauler: 20
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 17
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0
Torn: MIN
Tauler: 16
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 13
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 12
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 9
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 8
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 5
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 4
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 1
Moviments: [1]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 0
Moviments: []
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 0
Moviments: []
Moviment: 1
Utilitat: 0



1

Como se observa, la poda alfa-beta mejora mucho el rendimiento de minimax. En esta prueba, el algoritmo con poda alfa-beta es más de diez veces más rápido:

* Minimax: 42,7 s ± 1,33 s por ejecución (media ± desviación estándar de 7 ejecuciones, una iteración cada una).
* Minimax con poda alfa-beta: 3,41 s ± 124 ms por ejecución (media ± desviación estándar de 7 ejecuciones, una iteración cada una).

## Minimax con profundidad limitada y evaluación heurística

Aunque la poda alfa-beta mejora el rendimiento, minimax sigue siendo poco práctico para juegos con muchas posibilidades. Limitaremos la profundidad de búsqueda y, al alcanzarla, evaluaremos el estado con una función heurística.

Este algoritmo no garantiza encontrar el mejor movimiento, pero es más eficiente y permite jugar a juegos con más posibilidades.

La función heurística influye mucho en la calidad del algoritmo: debe estimar quién ganará y asignar valores altos a las situaciones favorables para un jugador y valores bajos a las desfavorables.

Para implementar minimax con poda alfa-beta y profundidad limitada necesitaremos:

* `maximitza(joc, estat, alfa, beta, profunditat)`: busca el movimiento más conveniente para `'MAX'` hasta la profundidad indicada. `alfa` y `beta` recogen las mejores opciones encontradas para `'MAX'` y `'MIN'`.
* `minimitza(joc, estat, alfa, beta, profunditat)`: busca el movimiento de menor utilidad para `'MAX'` hasta la profundidad indicada.
* `minimax_ab_prof(joc, estat, profunditat)`: devuelve el movimiento con mayor utilidad para `'MAX'` dentro de ese límite.
* `avaluacio_heuristica(joc, estat)`: estima la utilidad del estado para `'MAX'` mediante una heurística.
* `minimax_ab_prof_h(joc, estat, profunditat)`: utiliza la heurística al evaluar estados que alcanzan el límite de profundidad.

In [374]:
def avaluacio_heuristica(joc, estat, profunditat):
    if joc.test_terminal(estat):
        return joc.utilitat(estat, joc.torn(joc.inicial))
    if profunditat == 0:
        return 0
    v = -float('inf')
    for accio in joc.accions(estat):
        v = max(v, avaluacio_heuristica(joc, joc.resultat(estat, accio), profunditat - 1))
    return v

def minimax_ab_prof_h(joc, estat, profunditat=3):
    def maximitza(joc, estat, alfa, beta, profunditat):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        if profunditat == 0:
            print(avaluacio_heuristica(joc, estat, profunditat))
            return avaluacio_heuristica(joc, estat, profunditat), None
        v = -float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = minimitza(joc, joc.resultat(estat, accio), alfa, beta, profunditat - 1)
            if v2 > v:
                v = v2
                moviment = accio
            if v >= beta:
                return v, moviment
            alfa = max(alfa, v)
        return v, moviment

    def minimitza(joc, estat, alfa, beta, profunditat):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        if profunditat == 0:
            return avaluacio_heuristica(joc, estat, profunditat), None
        v = float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = maximitza(joc, joc.resultat(estat, accio), alfa, beta, profunditat - 1)
            if v2 < v:
                v = v2
                moviment = accio
            if v <= alfa:
                return v, moviment
            beta = min(beta, v)
        return v, moviment

    return maximitza(joc, estat, -float('inf'), float('inf'), profunditat)[1]

Comprobamos el algoritmo con una profundidad de 3. Compararemos `minimax_ab_prof_h` y `minimax_ab_prof` en 10 partidas de 25 fichas para evaluar la heurística.

In [375]:
results = []
for i in range(10):
    joc = JocNim(n_fitxes=25)
    result = joc.juga_joc(minimax_ab_prof_h, minimax_ab)
    results.append(result)

wins = [result == 1 for result in results]
print('Heuristic wins:', sum(wins))

Torn: MAX
Tauler: 25
Moviments: [1, 2, 3]
Moviment: None
Utilitat: 0

Torn: MIN
Tauler: 24
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0
Torn: MAX
Tauler: 21
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 20
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 17
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 16
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0
Torn: MAX
Tauler: 13
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 12
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 9
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 8
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 5
Moviments: [1, 2, 3]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 4
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 1
Moviments: [1]
Moviment: 3
Utilitat: 0

Torn: MIN
Tauler: 0
Moviments: []
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 0
Moviments: []
Moviment: 1
Utilitat: 0


Torn: MAX
Tauler: 25

Hemos ganado las 10 partidas con la función heurística. Esto sugiere que es eficaz en este ejemplo y puede ayudarnos a abordar juegos más complejos. Como prueba, podemos jugar una partida con 100 fichas.

In [376]:
joc = JocNim(n_fitxes=100)
joc.juga_joc(minimax_ab_prof_h, minimax_ab_prof_h)

Torn: MAX
Tauler: 100
Moviments: [1, 2, 3]
Moviment: None
Utilitat: 0

Torn: MIN
Tauler: 99
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 98
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 97
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 96
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 95
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 94
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 93
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 92
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 91
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 90
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 89
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 88
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 87
Moviments: [1, 2, 3]
Moviment: 1
Utilitat: 0

Torn: MAX
Tauler: 86
Moviments: [1, 2, 3]
Moviment: 1
Util

1

## Jugador humano

¿Puedes ganar a la máquina? Implementa un jugador humano y juega una partida contra ella.

In [382]:
def jugador_huma(joc, estat):
    moviment = None
    while moviment not in joc.accions(estat):
        moviment = int(input(f'Hi ha {estat.tauler} fitxes. Quantes vols treure?{joc.accions(estat)} '))
    return moviment

joc = JocNim(n_fitxes=100, moviments=[1, 2, 3, 5, 10])
joc.juga_joc(jugador_huma, minimax_ab_prof_h)

Torn: MAX
Tauler: 100
Moviments: [1, 2, 3, 5, 10]
Moviment: None
Utilitat: 0
Torn: MIN
Tauler: 90
Moviments: [1, 2, 3, 5, 10]
Moviment: 10
Utilitat: 0

Torn: MAX
Tauler: 89
Moviments: [1, 2, 3, 5, 10]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 79
Moviments: [1, 2, 3, 5, 10]
Moviment: 10
Utilitat: 0

Torn: MAX
Tauler: 78
Moviments: [1, 2, 3, 5, 10]
Moviment: 1
Utilitat: 0
Torn: MIN
Tauler: 68
Moviments: [1, 2, 3, 5, 10]
Moviment: 10
Utilitat: 0

Torn: MAX
Tauler: 67
Moviments: [1, 2, 3, 5, 10]
Moviment: 1
Utilitat: 0

Torn: MIN
Tauler: 57
Moviments: [1, 2, 3, 5, 10]
Moviment: 10
Utilitat: 0

Torn: MAX
Tauler: 56
Moviments: [1, 2, 3, 5, 10]
Moviment: 1
Utilitat: 0
Torn: MIN
Tauler: 46
Moviments: [1, 2, 3, 5, 10]
Moviment: 10
Utilitat: 0

Torn: MAX
Tauler: 45
Moviments: [1, 2, 3, 5, 10]
Moviment: 1
Utilitat: 0
Torn: MIN
Tauler: 35
Moviments: [1, 2, 3, 5, 10]
Moviment: 10
Utilitat: 0

Torn: MAX
Tauler: 34
Moviments: [1, 2, 3, 5, 10]
Moviment: 1
Utilitat: 0
Torn: MIN
Tauler: 24
Moviments: [

1